# Template Method Design Pattern 

explained using the classic Hot Beverage Maker (Tea vs Coffee) example.

#### The Concept

The Template Method defines the **skeleton of an algorithm** in a base class but lets subclasses override specific steps of the algorithm without changing its structure. 

**Analogy**: A Baking Recipe. The recipe (Template) says: "1. Mix Ingredients, 2. Bake, 3. Slice".
- **Cake**: Mixes flour/sugar, Bakes 30 mins, Slices into wedges.
- **Bread**: Mixes flour/water, Bakes 60 mins, Slices into squares. The steps are the same, but the details differ.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we use an **Abstract Base Class**. We mark the template method as `final` (so subclasses can't change the algorithm structure) and declare the specific steps as `abstract` (forcing subclasses to implement them).

#### THE ABSTRACT BASE (The Template)

In [3]:
from abc import ABC, abstractmethod

class CaffeineBeverage(ABC):
    
    # This is the TEMPLATE METHOD.
    # It controls the sequence.
    # We use a naming convention like 'final_prepare' to imply it shouldn't be overridden.
    def prepare_recipe(self):
        self.boil_water()
        self.brew()           # Abstract (Varies)
        self.pour_in_cup()
        if self.wants_condiments(): # Hook
            self.add_condiments() # Abstract (Varies)

    # --- Concrete Steps (Shared by all) ---
    def boil_water(self):
        print("🔥 Boiling water")

    def pour_in_cup(self):
        print("☕ Pouring into cup")

    # --- Abstract Steps (Must be implemented) ---
    @abstractmethod
    def brew(self):
        pass

    @abstractmethod
    def add_condiments(self):
        pass

    # --- Hook (Optional Override) ---
    # A hook has a default implementation that subclasses can ignore.
    def wants_condiments(self):
        return True

#### CONCRETE SUBCLASSES

In [4]:
class Tea(CaffeineBeverage):
    def brew(self):
        print("🍵 Steeping the tea")

    def add_condiments(self):
        print("🍋 Adding lemon")

class Coffee(CaffeineBeverage):
    def brew(self):
        print("💧 Dripping coffee through filter")

    def add_condiments(self):
        print("🥛 Adding sugar and milk")
    
    # Overriding the hook
    def wants_condiments(self):
        return False # Black coffee only

#### CLIENT CODE

In [5]:
def main():
    print("--- Making Tea ---")
    my_tea = Tea()
    my_tea.prepare_recipe()

    print("\n--- Making Coffee ---")
    my_coffee = Coffee()
    my_coffee.prepare_recipe()

if __name__ == "__main__":
    main()

--- Making Tea ---
🔥 Boiling water
🍵 Steeping the tea
☕ Pouring into cup
🍋 Adding lemon

--- Making Coffee ---
🔥 Boiling water
💧 Dripping coffee through filter
☕ Pouring into cup


## The Pythonic Way (Functional / Higher-Order Functions)

In Python, if the class doesn't hold much state (variables), creating a hierarchy just to change a few steps is overkill. We can use a **Higher-Order Function**. We define the "Template" as a function that accepts other functions (the steps) as arguments.

This turns the Template Method into a lightweight **Strategy Pattern**, which is often more Pythonic for simple algorithms.

#### THE TEMPLATE (A Function, not a Class)

In [6]:
from typing import Callable

def make_beverage(brew_algo: Callable, condiment_algo: Callable = None):
    """
    The skeleton of the algorithm is defined here.
    """
    print("🔥 Boiling water")
    
    # Call the passed function (The variable step)
    brew_algo()
    
    print("☕ Pouring into cup")
    
    if condiment_algo:
        condiment_algo()
    else:
        print("🚫 No condiments requested")

#### THE STEPS (Simple Functions)

In [7]:
def steep_tea():
    print("🍵 Steeping the tea")

def add_lemon():
    print("🍋 Adding lemon")

def drip_coffee():
    print("💧 Dripping coffee")

In [8]:
def main():
    print("--- Pythonic Tea ---")
    # We compose the algorithm on the fly
    make_beverage(brew_algo=steep_tea, condiment_algo=add_lemon)

    print("\n--- Pythonic Black Coffee ---")
    # We can omit the condiment step easily
    make_beverage(brew_algo=drip_coffee)

    print("\n--- Custom Drink (Lambda) ---")
    # We can even define steps inline!
    make_beverage(
        brew_algo=lambda: print("🍫 Mixing Hot Chocolate"),
        condiment_algo=lambda: print("🍦 Adding Marshmallows")
    )

if __name__ == "__main__":
    main()

--- Pythonic Tea ---
🔥 Boiling water
🍵 Steeping the tea
☕ Pouring into cup
🍋 Adding lemon

--- Pythonic Black Coffee ---
🔥 Boiling water
💧 Dripping coffee
☕ Pouring into cup
🚫 No condiments requested

--- Custom Drink (Lambda) ---
🔥 Boiling water
🍫 Mixing Hot Chocolate
☕ Pouring into cup
🍦 Adding Marshmallows


#### Key Differences

| Feature        | Classic OOP                                                     | Pythonic (Functional)                                           |
|----------------|-----------------------------------------------------------------|------------------------------------------------------------------|
| **Structure**  | Inheritance hierarchy (`Tea` extends `Beverage`).               | Functional composition (pass functions as arguments).            |
| **Enforcement**| Strong — `@abstractmethod` enforces implementation.             | Loose — any callable can be passed.                              |
| **Use Case**   | Complex algorithms with shared internal state (fields).         | Best suited for simpler behavior composition and pipelines.      |


#### When to use which?

- **Java Way**: Use this when the steps share data (e.g., `self.temperature`, `self.water_amount`) stored in the parent class.
- **Pythonic Way**: Use this for simple workflows (e.g., "Open File -> `Parse` -> Close File") where the parsing logic is the only thing changing and doesn't need to be a class.

# Template Method Design Pattern 

explained using a complex, real-world example: An ETL (Extract, Transform, Load) Data Pipeline.

#### The Scenario: Data Ingestion System

Your company receives data files from different vendors (CSV, JSON, XML). You need to ingest them into a central database. The **Overall Process (The Template)** is always the same:
- **Open File**: (Standard)
- **Extract Raw Data**: (Varies by format: Split text vs JSON parse)
- **Transform/Clean**: (Varies: Format dates, remove nulls)
- **Load to DB**: (Standard: SQL Insert)
- **Notify Admin**: (Hook: Optional, usually emails on failure)
- **Close File**: (Standard)

The "Skeleton" is fixed to ensure resource safety (always close files) and consistency (always save to DB), but the parsing logic is unique.

## The Classic OOP Way (Java-Style)

We use an **Abstract Base Class**. We use final (simulated in Python via comments or naming convention) to prevent subclasses from breaking the pipeline structure. We define `abstract` methods for the parts that must change.

#### THE ABSTRACT BASE (The Pipeline)

In [10]:
from abc import ABC, abstractmethod
from typing import List, Any

class DataMiner(ABC):
    
    # THE TEMPLATE METHOD
    # It defines the strict sequence of execution.
    def mine_data(self, path: str):
        file = self.open_file(path)
        
        try:
            raw_data = self.extract_data(file)
            clean_data = self.transform_data(raw_data)
            self.load_to_db(clean_data)
            
            # Hook: Subclasses can override if they want alerts
            if self.should_notify():
                self.send_report()
                
        except Exception as e:
            print(f"❌ Error occurred: {e}")
        finally:
            self.close_file(file)

    # --- Standard Steps (Shared) ---
    def open_file(self, path: str):
        print(f"\n📂 Opening file: {path}")
        return f"FileHandle({path})"

    def load_to_db(self, data: List[Any]):
        print(f"💾 Inserting {len(data)} records into SQL Database...")

    def close_file(self, file):
        print(f"🔒 Closing resource: {file}")

    def send_report(self):
        print("📧 Sending success email to admin...")

    # --- Abstract Steps (Forced Override) ---
    @abstractmethod
    def extract_data(self, file) -> Any:
        pass

    @abstractmethod
    def transform_data(self, data: Any) -> List[Any]:
        pass

    # --- Hooks (Optional Override) ---
    def should_notify(self):
        return False # Default is silent

#### CONCRETE IMPLEMENTATIONS

In [11]:
class CSVDataMiner(DataMiner):
    def extract_data(self, file) -> Any:
        print("   🔨 Extracting CSV: Splitting by commas...")
        return ["row1,john,doe", "row2,jane,smith"]

    def transform_data(self, data: Any) -> List[Any]:
        print("   ✨ Transforming CSV: converting strings to objects...")
        return [{"id": 1, "name": "John"}, {"id": 2, "name": "Jane"}]

    # We want notifications for CSVs
    def should_notify(self):
        return True

class JSONDataMiner(DataMiner):
    def extract_data(self, file) -> Any:
        print("   🔨 Extracting JSON: Parsing brackets...")
        return {"users": [{"id": 99, "val": "high"}, {"id": 100, "val": "low"}]}

    def transform_data(self, data: Any) -> List[Any]:
        print("   ✨ Transforming JSON: Flattening hierarchy...")
        return data["users"]

#### CLIENT CODE

In [12]:
def main():
    print("--- Job 1: CSV Processing ---")
    csv_miner = CSVDataMiner()
    csv_miner.mine_data("data.csv")

    print("\n--- Job 2: JSON Processing ---")
    json_miner = JSONDataMiner()
    json_miner.mine_data("data.json")

if __name__ == "__main__":
    main()

--- Job 1: CSV Processing ---

📂 Opening file: data.csv
   🔨 Extracting CSV: Splitting by commas...
   ✨ Transforming CSV: converting strings to objects...
💾 Inserting 2 records into SQL Database...
📧 Sending success email to admin...
🔒 Closing resource: FileHandle(data.csv)

--- Job 2: JSON Processing ---

📂 Opening file: data.json
   🔨 Extracting JSON: Parsing brackets...
   ✨ Transforming JSON: Flattening hierarchy...
💾 Inserting 2 records into SQL Database...
🔒 Closing resource: FileHandle(data.json)


## The Pythonic Way (Composition & Functional)

While the class-based approach is perfectly valid in Python, a more "Pythonic" approach for ETL pipelines often leverages **functional composition** or passing **callables** to a generic processor.

However, if we want to stick to the pattern but make it cleaner, we can use a **Context Manager** (**with** statement) to handle the Open/Close logic automatically (which is part of the template) and functions for the variable logic.

Here is a version using **Context Managers** and **Functional Strategy** (often preferred over rigid inheritance for pipelines).

#### THE TEMPLATE (Context Manager + Runner)

In [13]:
from contextlib import contextmanager
from typing import Callable, List, Any

class ETLEngine:
    def __init__(self, path: str):
        self.path = path

    # The "Open/Close" part of the template is best handled by __enter__/__exit__
    def __enter__(self):
        print(f"\n📂 [System] Opening {self.path}...")
        return self.path # Simulating file handle

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"🔒 [System] Closing {self.path}...")
        if exc_type:
            print(f"❌ [System] Error handled: {exc_val}")
        return True # Suppress errors

    # THE TEMPLATE METHOD
    # Instead of forcing inheritance, we accept logic as arguments (Strategy)
    def process(self, 
                extract_func: Callable[[str], Any], 
                transform_func: Callable[[Any], List[Any]]):
        
        # 1. We use our own context manager for the resource safety template
        with self as file_handle:
            # 2. Run the variable steps
            raw = extract_func(file_handle)
            clean = transform_func(raw)
            
            # 3. Run the common step
            self._load_db(clean)

    def _load_db(self, data):
        print(f"💾 [DB] Saved {len(data)} records.")

#### THE STEPS (Just Functions)

In [14]:
# Look how much easier it is to test these functions in isolation!

def parse_csv(file_handle):
    print("   🔨 CSV Logic: Split lines")
    return ["a,b", "c,d"]

def clean_csv(raw_data):
    print("   ✨ CSV Clean: UpperCase")
    return [x.upper() for x in raw_data]

def parse_json(file_handle):
    print("   🔨 JSON Logic: Decode object")
    return {"items": [1, 2, 3]}

def clean_json(raw_data):
    print("   ✨ JSON Clean: Filter valid")
    return raw_data["items"]

#### CLIENT CODE

In [15]:
def main():
    # 1. CSV Job
    engine_csv = ETLEngine("sales.csv")
    
    # We just plug in the functions we want. 
    # No need to create a new class "CSVDataMiner"
    engine_csv.process(
        extract_func=parse_csv, 
        transform_func=clean_csv
    )

    # 2. JSON Job
    engine_json = ETLEngine("users.json")
    
    engine_json.process(
        extract_func=parse_json, 
        transform_func=clean_json
    )

if __name__ == "__main__":
    main()


📂 [System] Opening sales.csv...
   🔨 CSV Logic: Split lines
   ✨ CSV Clean: UpperCase
💾 [DB] Saved 2 records.
🔒 [System] Closing sales.csv...

📂 [System] Opening users.json...
   🔨 JSON Logic: Decode object
   ✨ JSON Clean: Filter valid
💾 [DB] Saved 3 records.
🔒 [System] Closing users.json...


#### Why the Pythonic version is often preferred

- **Testing**: In the OOP version, to test the `transform_data` logic for CSV, you have to instantiate the whole `CSVDataMiner` class. In the Pythonic version, `clean_csv` is just a standalone function. You can test it with assert `clean_csv(["a"]) == ["A"]` instantly.
- **Reduction of Classes**: Real-world ETL systems might handle 50 formats. In OOP, that's 50 new classes. In Python, that's just a module with 50 small functions.
- **Context Managers**: Using `__enter__` and `__exit__` is the standard Python way to enforce the "Startup/Teardown" part of a template algorithm.